In [ ]:
import torch
import torchvision
from torch import nn

In [ ]:
# define image transforms (preprocessing steps)
image_transforms = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    # images are normalized to the range [0, 1]
])

# define training and test datasets, supply image transforms
# docs: https://pytorch.org/vision/0.19/generated/torchvision.datasets.FashionMNIST.html
train_ds = torchvision.datasets.FashionMNIST(root="./fashion_mnist", train=True, download=True, transform=image_transforms)
test_ds = torchvision.datasets.FashionMNIST(root="./fashion_mnist", train=False, download=True, transform=image_transforms)

# define torch.utils.data.DataLoader
# DataLoader docs: https://pytorch.org/tutorials/beginner/basics/data_tutorial.html#preparing-your-data-for-training-with-dataloaders
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=64, shuffle=False)

In [ ]:
from torchinfo import summary

# Docs for,
# Conv2d: https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
# LeakyReLU: https://pytorch.org/docs/stable/generated/torch.nn.LeakyReLU.html
# MaxPool2d: https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html
# Flatten: https://pytorch.org/docs/stable/generated/torch.nn.Flatten.html
# Linear: https://pytorch.org/docs/stable/generated/torch.nn.Linear.html
model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, stride=1),
    nn.LeakyReLU(),
    nn.MaxPool2d(kernel_size=2, stride=1),
    nn.Conv2d(16, 32, kernel_size=3, stride=1),
    nn.LeakyReLU(),
    nn.MaxPool2d(kernel_size=2, stride=1),
    nn.Flatten(),
    nn.Linear(15488, 128),
    nn.LeakyReLU(),
    nn.Linear(128, 10)
)
print(summary(model, input_size=(64, 1, 28, 28)))

# (optional) use PyTorch's graph compilation strategy to speedup computations
# docs: https://pytorch.org/tutorials/intermediate/torch_compile_tutorial.html#
@torch.compile
def execute_model():
    inp = torch.randn(64, 1, 28, 28)
    for layer in model:
        inp = layer(inp)
execute_model()

In [ ]:
from sklearn.metrics import accuracy_score

# CrossEntropyLoss expects unnormalized logits as input
# docs: https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html
ce_loss = torch.nn.CrossEntropyLoss()

# the adam optimizer
# docs: https://pytorch.org/docs/stable/generated/torch.optim.Adam.html
optimizier = torch.optim.Adam(model.parameters())

# define training and testing steps
def train_step(batch):
    batch_X, batch_y = batch
    optimizier.zero_grad()
    logits = model(batch_X)
    batch_loss = ce_loss(logits, batch_y)
    batch_loss.backward()
    optimizier.step()
    
    # compute accuracy for the batch
    pred_labels = torch.argmax(logits, axis=1)
    target_labels = batch_y
    batch_acc = accuracy_score(target_labels, pred_labels)
    
    return batch_loss.item(), batch_acc

def test_step(batch):
    batch_X, batch_y = batch
    logits = model(batch_X)
    batch_loss = ce_loss(logits, batch_y)
    
    # compute accuracy for the batch
    pred_labels = torch.argmax(logits, axis=1)
    target_labels = batch_y
    batch_acc = accuracy_score(target_labels, pred_labels)
    
    return batch_loss.item(), batch_acc
    
def epoch():
    mean_train_loss = 0.0
    mean_train_acc = 0.0
    for i, batch in enumerate(train_loader):
        batch_train_loss, batch_train_acc = train_step(batch)
        mean_train_loss += batch_train_loss
        mean_train_acc += batch_train_acc
    mean_train_loss = mean_train_loss / len(train_loader)
    mean_train_acc = mean_train_acc / len(train_loader)
        
    mean_test_loss = 0.0
    mean_test_acc = 0.0
    for i, batch in enumerate(test_loader):
        batch_test_loss, batch_test_acc = test_step(batch)
        mean_test_loss += batch_test_loss
        mean_test_acc += batch_test_acc
    mean_test_loss = mean_test_loss / len(test_loader)
    mean_test_acc = mean_test_acc / len(test_loader)
    
    return mean_train_loss, mean_train_acc, mean_test_loss, mean_test_acc

In [ ]:
n_epochs = 5
for _ in range(n_epochs):
    train_loss, train_acc, test_loss, test_acc = epoch()
    print(f"train_loss: {train_loss}\ttest_loss: {test_loss}\ttrain_acc: {train_acc}\ttest_acc: {test_acc}")